# Table 1: crisis overview

Columns C1 ... C7.  First row: duration of the crisis in S&P 500 trading days from the peak to the trough (crises.py).  Then the maximum drawdown within each crisis (S&P 500 peak -> back at the peak) for GSPC, MCap, 1/N, HRP, MVO, LSTM, RF, XGB (baselines, monthly).

Output: `data_finalized/table_1_crisis_overview.csv`

In [ ]:
'''
Table 1: crisis overview.  Duration per crisis (S&P 500 trading days peak ->
trough), then the maximum drawdown within each crisis (S&P 500 peak -> back
at the peak) per portfolio.
Portfolios: sheets C1 ... C7 of crisis_analysis_all_monthly.xlsx (column
full_max_dd, crisis_analysis_defined.py).  GSPC is not in that workbook
(sp500 is skipped there), so its row is computed with the same window_row().
Written for reading: days as whole numbers, drawdowns in % with one decimal.
'''

import sys
from pathlib import Path

import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
import crisis_analysis_defined as cad
from crises import main_crises

FREQ     = 'monthly'
OUT_PATH = PAPER / '_latex' / 'data_finalized' / 'table_1_crisis_overview.csv'

# name in the table -> output folder (_output/<model>/<folder>), in table order
GSPC = ('GSPC', 's&p_500')
RUNS = {
    'MCap': 'mcap_t_10_monthly',
    '1/N':  'equal_t_10_monthly',
    'HRP':  'hrp_t_10_lw_ward_monthly',
    'MVO':  'mvo_t_10_lw_monthly',
    'LSTM': 'lstm_t_10_holdout_rank_monthly',
    'RF':   'rf_t_10_holdout_monthly',
    'XGB':  'xgb_t_10_holdout_monthly',
}

crisis_ids = [f'C{i}' for i in range(1, len(main_crises) + 1)]
windows    = {name: w for name, w, phases in cad.WINDOWS if phases}             # C1 ... C7

sp = next(r for r in ca.find_runs() if r['run'] == GSPC[1] and r['frequency'] == FREQ)

# ---- duration: S&P 500 trading days in (peak, trough]
days = {c: len(ca._slice(sp['returns'], windows[c]['peak'], windows[c]['trough'])) for c in crisis_ids}

# ---- GSPC: sp500 run, same window_row() as the workbook
dd = {GSPC[0]: {c: cad.window_row(sp, sp['returns'], windows[c], phases=True)['full_max_dd']
                for c in crisis_ids}}

# ---- portfolios: full_max_dd from the crisis sheets
sheets = pd.read_excel(cad.SAVE_DIR / f'crisis_analysis_all_{FREQ}.xlsx', sheet_name=crisis_ids)
for label, folder in RUNS.items():
    run = folder.removesuffix(f'_{FREQ}')
    vals = {}
    for c in crisis_ids:
        hit = sheets[c][(sheets[c]['run'] == run) & (sheets[c]['frequency'] == FREQ)]
        if hit.empty:
            raise KeyError(f'{folder!r} not in the {FREQ} {c} sheet')
        vals[c] = hit.iloc[0]['full_max_dd']
    dd[label] = vals

# ---- readable cells: days as integers, drawdowns x 100 with one decimal and '%'
rows = [pd.Series({c: f'{days[c]:d}' for c in crisis_ids}, name='days')]
rows += [pd.Series({c: f'{100 * v:.1f}%' for c, v in vals.items()}, name=label)
         for label, vals in dd.items()]
table_1 = pd.DataFrame(rows)[crisis_ids].rename_axis('model').reset_index()

OUT_PATH.parent.mkdir(exist_ok=True)
table_1.to_csv(OUT_PATH, index=False)
print(f'wrote {OUT_PATH}: {table_1.shape[0]} rows x {table_1.shape[1] - 1} columns')
table_1

# Table 2: full-period performance

Rows as in Table 1 (GSPC, MCap, 1/N, HRP, MVO, LSTM, RF, XGB; baselines, monthly).  Columns over the full period 1998-2025:

|sharpe_ratio|cagr|cum_return|annual_volatility|max_drawdown|ulcer_index|turnover_pa|

Output: `data_finalized/table_2_full_period.csv`

In [ ]:
'''
Table 2: full-period performance 1998-2025 (crises.overall).
Portfolios: overall sheet of crisis_analysis_all_monthly.xlsx
(crisis_analysis_defined.py).  GSPC is not in that workbook (sp500 is
skipped there), so its row is computed with the same window_row().
Written for reading, decimals per column (COLUMNS).  GSPC is buy & hold:
no turnover after the initial purchase (0%).
'''

import sys
from pathlib import Path

import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
import crisis_analysis_defined as cad
from crises import overall

FREQ       = 'monthly'
OUT_PATH_2 = PAPER / '_latex' / 'data_finalized' / 'table_2_full_period.csv'

# name in the table -> output folder (_output/<model>/<folder>), in table order
GSPC = ('GSPC', 's&p_500')
RUNS = {
    'MCap': 'mcap_t_10_monthly',
    '1/N':  'equal_t_10_monthly',
    'HRP':  'hrp_t_10_lw_ward_monthly',
    'MVO':  'mvo_t_10_lw_monthly',
    'LSTM': 'lstm_t_10_holdout_rank_monthly',
    'RF':   'rf_t_10_holdout_monthly',
    'XGB':  'xgb_t_10_holdout_monthly',
}

# table column -> (source column, format); '{:.1%}' = x 100 with one decimal and '%'
COLUMNS = {
    'sharpe_ratio':      ('full_sharpe',      '{:.2f}'),
    'cagr':              ('full_cagr',        '{:.2%}'),
    'cum_return':        ('full_cum_return',  '{:.0%}'),
    'annual_volatility': ('full_ann_vol',     '{:.1%}'),
    'max_drawdown':      ('full_max_dd',      '{:.1%}'),
    'ulcer_index':       ('full_ulcer',       '{:.1%}'),
    'turnover_pa':       ('full_turnover_pa', '{:.0%}'),
}

# ---- GSPC: sp500 run, same window_row() as the workbook's overall sheet
sp = next(r for r in ca.find_runs() if r['run'] == GSPC[1] and r['frequency'] == FREQ)
values = {GSPC[0]: cad.window_row(sp, sp['returns'], overall[0], phases=False)}
values[GSPC[0]]['full_turnover_pa'] = 0.0               # buy & hold

# ---- portfolios: overall sheet
src = pd.read_excel(cad.SAVE_DIR / f'crisis_analysis_all_{FREQ}.xlsx', sheet_name='overall')
src = src[src['frequency'] == FREQ].set_index('run')
for label, folder in RUNS.items():
    run = folder.removesuffix(f'_{FREQ}')
    if run not in src.index:
        raise KeyError(f'{folder!r} not in the {FREQ} overall sheet')
    values[label] = src.loc[run]

table_2 = pd.DataFrame(
    [pd.Series({col: f.format(values[label][s]) for col, (s, f) in COLUMNS.items()}, name=label)
     for label in [GSPC[0], *RUNS]]).rename_axis('model').reset_index()

OUT_PATH_2.parent.mkdir(exist_ok=True)
table_2.to_csv(OUT_PATH_2, index=False)
print(f'wrote {OUT_PATH_2}: {table_2.shape[0]} rows x {table_2.shape[1] - 1} columns')
table_2

# Table 3: average crisis performance

Rows as in Table 1.  Each column is the equal-weighted average over the seven crises C1 ... C7:

|avg_max_dd|avg_ulcer|avg_recovery_days|avg_return_full|avg_return_ptt|avg_return_ttp|

Fixed S&P 500 windows: full = peak -> back at the peak, ptt = peak -> trough, ttp = trough -> back at the peak.  Max. DD and Ulcer index over the full window; cumulative return over each window.  Recovery: trading days from the portfolio's own trough until it is back at its value on the S&P peak date, searched up to the next crisis peak (C7: end of the data); a portfolio not recovered by then counts up to that cut-off, and its average is marked as a lower bound (≥).

Output: `data_finalized/table_3_crisis_average.csv`

In [ ]:
'''
Table 3: average crisis performance, equal weight over C1 ... C7.
Computed from each run's daily returns with the functions behind the crisis
workbooks, so GSPC and the portfolios are treated the same:
  max. DD, Ulcer       crisis_analysis_defined.full_metrics, fixed window
                       (S&P peak, S&P back at the peak]
  recovery             crisis_analysis_variable.episode: ttp_days = trading
                       days own trough -> back at the anchor (value on the S&P
                       peak date), searched up to the next crisis peak
                       (C7: end of the data); full_not_recovered = cut off
  cumulative return    metrics.cumulative_return over (peak, even],
                       (peak, trough], (trough, even]
Written for reading; an average that includes a cut-off recovery is a lower
bound and starts with '≥'.
'''

import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
import crisis_analysis_defined as cad
import crisis_analysis_variable as cav
from crises import main_crises
from metrics import cumulative_return

FREQ       = 'monthly'
OUT_PATH_3 = PAPER / '_latex' / 'data_finalized' / 'table_3_crisis_average.csv'

# name in the table -> output folder (_output/<model>/<folder>), in table order
RUNS = {
    'GSPC': 's&p_500',
    'MCap': 'mcap_t_10_monthly',
    '1/N':  'equal_t_10_monthly',
    'HRP':  'hrp_t_10_lw_ward_monthly',
    'MVO':  'mvo_t_10_lw_monthly',
    'LSTM': 'lstm_t_10_holdout_rank_monthly',
    'RF':   'rf_t_10_holdout_monthly',
    'XGB':  'xgb_t_10_holdout_monthly',
}

by_run = {r['run']: r for r in ca.find_runs() if r['frequency'] == FREQ}

def crisis_values(ret: pd.Series, i: int, c: dict) -> dict:
    '''All Table 3 metrics of one run in crisis c (index i in main_crises).'''
    s = lambda a, b: ca._slice(ret, c[a], c[b])
    full = cad.full_metrics(s('peak', 'even'), pd.Series(dtype=float))
    rec_end = main_crises[i + 1]['peak'] if i + 1 < len(main_crises) else None
    ep = cav.episode(ret, c, rec_end)
    return {
        'max_dd':        full['full_max_dd'],
        'ulcer':         full['full_ulcer'],
        'recovery_days': ep['ttp_days'],
        'censored':      ep['full_not_recovered'],
        'return_full':   cumulative_return(s('peak', 'even')),
        'return_ptt':    cumulative_return(s('peak', 'trough')),
        'return_ttp':    cumulative_return(s('trough', 'even')),
    }

rows, censored = [], {}
for label, folder in RUNS.items():
    run = folder.removesuffix(f'_{FREQ}')
    if run not in by_run:
        raise KeyError(f'no {FREQ} run {folder!r}')
    per = pd.DataFrame([crisis_values(by_run[run]['returns'], i, c) for i, c in enumerate(main_crises)],
                       index=[f'C{i}' for i in range(1, len(main_crises) + 1)])
    censored[label] = per.index[per['censored']].tolist()
    avg = per.drop(columns='censored').mean()
    low = '≥' if censored[label] else ''
    rows.append(pd.Series({
        'avg_max_dd':        f'{avg["max_dd"]:.1%}',
        'avg_ulcer':         f'{avg["ulcer"]:.1%}',
        'avg_recovery_days': f'{low}{avg["recovery_days"]:.0f}',
        'avg_return_full':   f'{avg["return_full"]:.1%}',
        'avg_return_ptt':    f'{avg["return_ptt"]:.1%}',
        'avg_return_ttp':    f'{avg["return_ttp"]:.1%}',
    }, name=label))

table_3 = pd.DataFrame(rows).rename_axis('model').reset_index()

OUT_PATH_3.parent.mkdir(exist_ok=True)
table_3.to_csv(OUT_PATH_3, index=False, encoding='utf-8')
print(f'wrote {OUT_PATH_3}: {table_3.shape[0]} rows x {table_3.shape[1] - 1} columns')
print('recovery cut off at the next crisis peak:', {k: v for k, v in censored.items() if v})
table_3

# Table 4: regime detectors

12 rows: changepoint (VIX), HMM (S&P 500), clustering (Wasserstein k-means, S&P 500 + VIX), grouped by daily / monthly / quarterly / yearly.  Daily = the detector's own daily state; monthly / quarterly / yearly = the state on the first trading day of the month / quarter / year, held until the next one (as the portfolios hold it).

|n_episodes|avg_episode_days|defensive_share|avg_entry_lag|avg_exit_lag|recall|precision|

Episodes: number of switches into the crisis state and their average length (trading days).  Entry lag: trading days from the S&P peak to the first crisis day (negative = already in crisis at the peak; averaged over the crises entered before the trough).  Exit lag: trading days from the S&P trough to the first calm day (averaged over the crises in which the detector switches back).  Number of crises behind each average lag in brackets.  Recall / precision per day against the label S&P peak -> trough.

Output: `data_finalized/table_4_detectors.csv`

In [ ]:
'''
Table 4: regime detectors, independent of the portfolio models.
States and metrics from detector_evaluation.py, recomputed here:
  daily      detector_evaluation.daily_state: crisis on t if the stored
             signal of t-1 is above regime_def.CRISIS_THRESHOLD
  monthly / quarterly / yearly
             detector_evaluation.applied_states: the crisis flag of the regime
             runs (regime.csv), set on the first trading day of the month /
             quarter / year and held from the next day until the next one
  per crisis crisis_metrics: exit_lag (S&P peak -> first crisis day; here the
             entry lag), reentry_lag (S&P trough -> first calm day; here the
             exit lag); NaN where the detector never switches
  full period  full_period: n_episodes, mean_episode_days,
             total_defensive_share, recall_ptt, precision_ptt
Calendar: the trading days 1998-01-02 .. 2025-12-31 of the portfolio runs.
The Wasserstein k-means detector is called "Clustering" in the paper.
'''

import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics')]
import compare_all as ca
import crisis_analysis_defined as cad
import detector_evaluation as de
from crises import overall, main_crises

OUT_PATH_4  = PAPER / '_latex' / 'data_finalized' / 'table_4_detectors.csv'
FREQUENCIES = ['daily', 'monthly', 'quarterly', 'yearly']
# (detector, series) -> name in the table, in table order
DETECTORS = {('changepoint', 'vix'): 'Changepoint',
             ('hmm', 'gspc'): 'HMM',
             ('wasserstein', 'gspc_vix'): 'Clustering'}

runs     = ca.find_runs()
bench    = next(r for r in runs if r['model'] == cad.BENCHMARK and r['frequency'] == 'monthly')
calendar = ca._slice(bench['returns'], overall[0]['peak'], overall[0]['even']).index
sp       = next(r for r in runs if r['model'] == de.SP500_MODEL)['returns']
applied  = de.applied_states(runs)

def lag(values: list) -> str:
    '''Average over the crises with a value, and how many that are.'''
    v = np.array(values, dtype=float)
    n = int(np.isfinite(v).sum())
    return f'{np.nanmean(v):.1f} ({n})' if n else f'-- (0)'

rows = []
for freq in FREQUENCIES:
    for (det, ser), name in DETECTORS.items():
        if freq == 'daily':
            d = de.daily_state(det, ser, calendar)
        else:
            if (det, ser, freq) not in applied:
                raise KeyError(f'no {freq} regime run for {det} {ser}')
            d = applied[(det, ser, freq)].reindex(calendar, fill_value=False)
        pc = [de.crisis_metrics(d, k) for k in range(len(main_crises))]
        full, _ = de.full_period(d, pc, sp, {})
        rows.append({
            'frequency':        freq,
            'detector':         name,
            'n_episodes':       f'{full["n_episodes"]:d}',
            'avg_episode_days': f'{full["mean_episode_days"]:.0f}',
            'defensive_share':  f'{full["total_defensive_share"]:.1%}',
            'avg_entry_lag':    lag([m['exit_lag'] for m in pc]),
            'avg_exit_lag':     lag([m['reentry_lag'] for m in pc]),
            'recall':           f'{full["recall_ptt"]:.1%}',
            'precision':        f'{full["precision_ptt"]:.1%}',
        })

table_4 = pd.DataFrame(rows)

OUT_PATH_4.parent.mkdir(exist_ok=True)
table_4.to_csv(OUT_PATH_4, index=False, encoding='utf-8')
print(f'wrote {OUT_PATH_4}: {table_4.shape[0]} rows x {table_4.shape[1] - 2} columns')
table_4

# Figure: monthly crisis probability of the detectors

One row per month 1998-01 ... 2025-12: the first trading day of the month and the crisis probability $p_{crisis}$ each detector gives on that day (the value the monthly regime runs read on their rebalancing date; it uses data through the day before).  Columns in chart order, top to bottom:

|date|changepoint|clustering|hmm|

Changepoint = VIX, clustering = Wasserstein k-means on S&P 500 + VIX, HMM = S&P 500.  Checked: $p_{crisis} > 0.5$ equals the crisis flag of the monthly regime runs on every date.

Output: `data_finalized/figure_pcrisis_monthly.csv`

In [ ]:
'''
Figure: monthly crisis probability of the three detectors, for the stacked
line chart in paper.tex (top to bottom: changepoint, clustering, HMM).
Dates: first trading day of each month in the calendar of the portfolio runs
(1998-01-02 .. 2025-12-31), i.e. the monthly rebalancing dates.
Value: regime_def._signal, forward-filled to that day, exactly as
regime_def.regime_probs reads it on a rebalancing date (stored with a one-day
shift, so it uses data through the day before).
Check: p_crisis > CRISIS_THRESHOLD equals the crisis flag (regime.csv) of the
monthly regime runs on every date.
'''

import sys
from pathlib import Path

import pandas as pd

HERE  = Path.cwd()                              # _latex, or the workspace root
PAPER = HERE.parent if (HERE.parent / '_comparison').is_dir() else HERE / 'Paper'
sys.path += [str(PAPER / '_comparison'), str(PAPER / '_metrics'), str(PAPER / '_regimes')]
import compare_all as ca
import crisis_analysis_defined as cad
import regime_def
from crises import overall

OUT_PATH_F = PAPER / '_latex' / 'data_finalized' / 'figure_pcrisis_monthly.csv'
FREQ = 'monthly'
# column -> (detector, series), in chart order (top to bottom)
DETECTORS = {'changepoint': ('changepoint', 'vix'),
             'clustering':  ('wasserstein', 'gspc_vix'),
             'hmm':         ('hmm', 'gspc')}

runs     = ca.find_runs()
bench    = next(r for r in runs if r['model'] == cad.BENCHMARK and r['frequency'] == FREQ)
calendar = ca._slice(bench['returns'], overall[0]['peak'], overall[0]['even']).index
dates    = calendar.to_series().groupby(calendar.to_period('M')).first()

figure = pd.DataFrame({'date': dates.dt.strftime('%Y-%m-%d').values})
for col, (det, ser) in DETECTORS.items():
    sig = regime_def._signal(det, ser, regime_def.SMOOTH_HALFLIFE)['p_crisis']
    p = sig.reindex(dates.values, method='ffill')
    if p.isna().any():
        raise ValueError(f'{col}: no signal on {p.index[p.isna()][0].date()}')
    run = next((r for r in runs if (r['detector'], r['series'], r['frequency']) == (det, ser, FREQ)), None)
    if run is None:
        raise KeyError(f'no {FREQ} regime run for {det} {ser}')
    flag = run['crisis'].reindex(dates.values, method='ffill') > regime_def.CRISIS_THRESHOLD
    if not ((p > regime_def.CRISIS_THRESHOLD) == flag).all():
        raise AssertionError(f'{col}: p_crisis does not match the {FREQ} regime run on '
                             f'{((p > regime_def.CRISIS_THRESHOLD) != flag).sum()} dates')
    figure[col] = p.round(4).values

OUT_PATH_F.parent.mkdir(exist_ok=True)
figure.to_csv(OUT_PATH_F, index=False)
print(f'wrote {OUT_PATH_F}: {len(figure)} months, {figure["date"].iloc[0]} .. {figure["date"].iloc[-1]}')
print('months in crisis (p > 0.5):', (figure[list(DETECTORS)] > regime_def.CRISIS_THRESHOLD).sum().to_dict())
figure